# ufcscraper BFO verification

**Source:** BestFightOdds.com (via the ufcscraper PyPI package, odds_scraper submodule)

**Package:** ufcscraper

**Run date:** 2026-05-04

This notebook verifies that BestFightOdds is a usable source before any commitment is made to it. It confirms the ufcscraper package installs, locates the BestFightOddsScraper class within the package structure, inspects its constructor and public methods, and documents the operational constraints (the Selenium dependency, captcha handling, and historical date range) that determine how the source could be used.

**Use in project:** A candidate validation source for the competitive axis. Implied win probability from pre-fight odds would give a market belief comparator for the Glicko-2 ratings, independent of a ranking system. This notebook verifies the source; its use is deferred to future work, and the competitive axis is validated against Fight Matrix instead.

## Section 1: Setup and imports

In [ ]:
# BLOCK 1: Setup

# ufcscraper BFO source verification

import ufcscraper
import os
import pandas as pd
import time
from datetime import datetime, timedelta
!pip install ufcscraper -q

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Display settings for readable dataframe output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 60)

print(f"Verification Run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Verification Run date: 2026-05-04 15:46:28


## Section 2: Verification

In [ ]:
# BLOCK 2: Source ufcscraper (BestFightOdds)

print("SOURCE 1: ufcscraper (BestFightOdds)")
print()

try:
    import ufcscraper
    print(f"OK ufcscraper imported successfully")
    print(f"   Package version: {ufcscraper.__version__ if hasattr(ufcscraper, '__version__') else 'version not exposed'}")
except ImportError as e:
    print(f"FAIL ufcscraper import failed: {e}")

print()
print("Available top-level attributes:")
print(dir(ufcscraper))

SOURCE 1: ufcscraper (BestFightOdds)

OK ufcscraper imported successfully
   Package version: version not exposed

Available top-level attributes:
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [ ]:
# BLOCK 3: Discover ufcscraper submodule structure

# locate the BFO scraper inside the odds_scraper submodule.

# Find the package installation directory
package_path = os.path.dirname(ufcscraper.__file__)
print(f"ufcscraper package location: {package_path}")
print()

# List all Python files in the package directory
print("Files in the package:")
for item in sorted(os.listdir(package_path)):
    full_path = os.path.join(package_path, item)
    if os.path.isfile(full_path) and item.endswith('.py'):
        print(f"  {item}")
    elif os.path.isdir(full_path) and not item.startswith('__'):
        print(f"  {item}/ (subdirectory)")

ufcscraper package location: /usr/local/lib/python3.12/dist-packages/ufcscraper

Files in the package:
  __init__.py
  base.py
  catch_weights.py
  event_scraper.py
  fight_scraper.py
  fighter_names.py
  fighter_scraper.py
  odds_scraper/ (subdirectory)
  replacement_scraper.py
  scripts/ (subdirectory)
  ufc_scraper.py
  utils.py


In [ ]:
# BLOCK 4: Inspect the odds_scraper subdirectory

# Path to the odds_scraper subdirectory
odds_scraper_path = os.path.join(package_path, 'odds_scraper')
print(f"odds_scraper subdirectory location: {odds_scraper_path}")
print()
print("Files in odds_scraper:")
for item in sorted(os.listdir(odds_scraper_path)):
    full_path = os.path.join(odds_scraper_path, item)
    if os.path.isfile(full_path) and item.endswith('.py'):
        print(f"  {item}")
    elif os.path.isdir(full_path) and not item.startswith('__'):
        print(f"  {item}/")

print()

init_path = os.path.join(odds_scraper_path, '__init__.py')
if os.path.exists(init_path):
    print("Contents of odds_scraper/__init__.py:")
    with open(init_path, 'r') as f:
        print(f.read())

odds_scraper subdirectory location: /usr/local/lib/python3.12/dist-packages/ufcscraper/odds_scraper

Files in odds_scraper:
  __init__.py
  bet365_odds_reader.py
  bfo_scraper.py

Contents of odds_scraper/__init__.py:
from .bfo_scraper import BestFightOddsScraper
from .bet365_odds_reader import Bet365OddsReader, Bet365Odds



In [ ]:
# BLOCK 5: Inspect the BestFightOddsScraper class

from ufcscraper.odds_scraper import BestFightOddsScraper
import os

# Inspect the class signature
import inspect
print("BestFightOddsScraper class signature:")
print(inspect.signature(BestFightOddsScraper.__init__))
print()

# Print the docstring
if BestFightOddsScraper.__init__.__doc__:
    print("Init docstring:")
    print(BestFightOddsScraper.__init__.__doc__)
print()

# List the methods available on the class
print("Public methods on BestFightOddsScraper:")
methods = [m for m in dir(BestFightOddsScraper) if not m.startswith('_') and callable(getattr(BestFightOddsScraper, m))]
for m in methods:
    method_obj = getattr(BestFightOddsScraper, m)
    sig = inspect.signature(method_obj) if callable(method_obj) else ''
    print(f"  {m}{sig}")

BestFightOddsScraper class signature:
(self, data_folder: 'Path | str', n_sessions: 'Optional[int]' = None, delay: 'Optional[float]' = None, min_score: 'Optional[int]' = None, min_date: 'datetime.date' = datetime.date(2008, 8, 1))

Init docstring:
Initialize the BestFightOddsScraper.

        It extends the BaseScraper class by adding score for
        naming matching, date filtering and initializes fighter_names
        class correspondent to the data_folder.

        Args:
            data_folder: Path to the folder where data is stored.
            n_sessions: Number of concurrent browser sessions.
            delay: Delay between requests.
            min_score: Minimum score for name matching.
            min_date: Minimum date for filtering events. Events prior to August
                2008 are not available.
        

Public methods on BestFightOddsScraper:
  captcha_indicator(driver: 'webdriver.Chrome') -> 'bool'
  check_data_file(self) -> 'None'
  create_search_url(query: 'st

## Verification summary

The package installs cleanly, and BestFightOddsScraper is accessible at ufcscraper.odds_scraper.BestFightOddsScraper. Its constructor accepts data_folder, n_sessions, delay, min_score, and min_date (defaulting to 1 August 2008, the BestFightOdds historical limit), and scrape_BFO_odds() is the primary scraping method. The class drives Selenium with Chrome, and BestFightOdds presents captchas that the package detects but does not solve.

No live scrape is run here. A captcha flow in a verification notebook adds nothing over confirming the package structure, and running Selenium from Colab is unreliable. Any first pull would run locally, with cached output and a one-time captcha cost, rather than in this notebook.

The underlying source is BestFightOdds itself; ufcscraper is one of several possible access paths, so the source remains reachable by direct scraping if the package becomes unmaintained. On this basis the source is verified as usable but its use is deferred to future work.